# Plot 5: CARROT vs binary routers on Open-LLM-Leaderboard v2

Binary baselines (route between Qwen2.5-7B and Qwen2.5-72B): RouteLLM MF, Not-Diamond RoRF.
CARROT variants (route across all 20 models): CARROT-KNN, CARROT-RoBERTa.

Cost on Open-LLM-v2 is computed as `input_tokens × $per-million-input` per model (no output-token cost modeled).

In [ ]:
import os, sys
sys.path.insert(0, '../carrot/')
import numpy as np
import matplotlib.pyplot as plt
from constants import OPEN_MODELS, OPEN_COSTS
from utils import route, route_pairwise

os.makedirs('../plots', exist_ok=True)
PREDS_DIR = '../data/open-llm-lb-v2/preds'

meta = np.load(f'{PREDS_DIR}/meta.npy', allow_pickle=True).item()
models = meta['models']
Y_test = meta['Y_test']
IT_test = meta['IT_test']
small_ind = meta['small_model_ind']
large_ind = meta['large_model_ind']

# Compute per-prompt actual cost from input tokens × model cost rates
mm = [m.replace('open-llm-leaderboard/', '').replace('__', '/').replace('-details', '') for m in OPEN_MODELS]
model_order = [np.argmax(np.array(models) == m) for m in mm if np.sum(np.array(models) == m) > 0]
cost_rates = np.array([OPEN_COSTS[m] / 1e6 for m in models])[None, :]
C_test = IT_test[:, model_order] * cost_rates
print(f'{len(models)} models; test n={Y_test.shape[0]}')

In [ ]:
def load(name):
    path = f'{PREDS_DIR}/{name}.npy'
    if not os.path.exists(path):
        print(f'MISSING: {path}')
        return None
    return np.load(path, allow_pickle=True)

Y_hat = {m: load(f'Y_hat_{m}') for m in ['mf', 'rorf', 'roberta-binary', 'carrot-knn', 'carrot-roberta']}

In [ ]:
curves = {}

for name in ['mf', 'rorf', 'roberta-binary']:
    if Y_hat[name] is None:
        continue
    c, p = route_pairwise(np.asarray(Y_hat[name]).squeeze(), C_test, Y_test, large_ind, small_ind)
    curves[name] = (c, p)

# For OpenLLM we don't model per-prompt cost prediction; use actual C_test as the routing cost signal
for name in ['carrot-knn', 'carrot-roberta']:
    if Y_hat[name] is None:
        continue
    c, p = route(Y_hat[name], C_test, C_test, Y_test)
    curves[name] = (c, p)

print('Curves produced:', list(curves.keys()))

In [ ]:
from matplotlib.ticker import MaxNLocator

labels = {'mf': 'RouteLLM (MF)', 'rorf': 'Not-Diamond RoRF', 'roberta-binary': 'RouteLLM (RoBERTa)',
          'carrot-knn': 'CARROT (KNN)', 'carrot-roberta': 'CARROT (RoBERTa)'}
colors = {'mf': 'blue', 'rorf': 'green', 'roberta-binary': 'purple',
          'carrot-knn': 'red', 'carrot-roberta': 'orange'}
markers = ['o', 's', 'D', '^', 'v', 'p', '*', 'x', '+', 'h', 'H', 'd', '>', 'P', '<', '|', '_', '.', ',', '1', '2']

LABEL_MODELS = {'Qwen/Qwen2.5-72B-Instruct', 'Qwen/Qwen2-72B-Instruct',
                'alpindale/WizardLM-2-8x22B', 'mistralai/Mistral-7B-Instruct-v0.3',
                'google/gemma-2b-it'}

fig, ax = plt.subplots(1, 1, figsize=(4.3, 4.3))
for name, (c, p) in curves.items():
    ax.errorbar(c, p, c=colors[name], linestyle='--', linewidth=1, label=labels[name])

for i, m in enumerate(models):
    x, y = C_test[:, i].mean(0), Y_test[:, i].mean(0)
    ax.scatter([x], [y], marker=markers[i % len(markers)])
    if m in LABEL_MODELS:
        ax.annotate(m.split('/')[-1], (x, y), size=6)

ax.set_title('Open-LLM-Leaderboard v2 — CARROT vs binary routers')
ax.set_xlabel('Cost Per Query, $')
ax.set_ylabel('Accuracy')
ax.set_xlim(left=0)
ax.xaxis.set_major_locator(MaxNLocator(nbins=3))
ax.legend()
ax.grid(True)
fig.savefig('../plots/openllm_binary.pdf', bbox_inches='tight')
plt.show()